# Pipeline KPI Surete Paris (version simple)

Objectif: construire un score de surete/risque par arrondissement de Paris et, en plus, un score agrege par zone IRIS.

Le pipeline reste simple:

1. chargement des fichiers
2. score delinquance par arrondissement
3. score par zone IRIS
4. agregation finale par arrondissement
5. export CSV

## Comment lire le pipeline

Le notebook suit toujours la meme logique:

1. on charge les fichiers
2. on garde Paris uniquement
3. on calcule un score de risque avec la delinquance
4. on ajoute les cameras et les commissariats
5. on obtient un score final de surete sur 100

Le score final est simple a lire:
- plus il est haut, plus le quartier est considere comme sur
- plus il est bas, plus le quartier est considere comme a risque

La partie la plus importante est la suivante:
- beaucoup de delinquance = risque plus eleve
- beaucoup de cameras = risque plus faible
- commissariat plus proche = risque plus faible

In [22]:
import pandas as pd
import numpy as np
from pathlib import Path

def minmax(series):
    smin = series.min()
    smax = series.max()
    if pd.isna(smin) or pd.isna(smax) or smin == smax:
        return pd.Series(0.5, index=series.index)
    return (series - smin) / (smax - smin)

def distance_km(lat1, lon1, lat2, lon2):
    dx = (lon2 - lon1) * 111 * np.cos(np.deg2rad(lat1))
    dy = (lat2 - lat1) * 111
    return np.sqrt(dx**2 + dy**2)

In [ ]:
# 1) Chargement des 4 datasets
path_delinquance = Path('data/bronze/surete/dataset_delinquances.csv')
path_cameras = Path('data/bronze/surete/points.csv')
path_commissariats = Path('data/bronze/surete/cartographie-des-emplacements-des-commissariats-a-paris-et-petite-couronne.csv')
path_iris = Path('data/bronze/commun/iris.csv')

df_delinq = pd.read_csv(path_delinquance, sep=';', decimal=',', quotechar='"', na_values=['NA'], low_memory=False)
df_cam = pd.read_csv(path_cameras, sep=',', low_memory=False)
df_comm = pd.read_csv(path_commissariats, sep=';', low_memory=False)
df_iris = pd.read_csv(path_iris, sep=';', low_memory=False)

print('Fichiers charges:')
print(f'- Delinquance: {len(df_delinq):,} lignes'.replace(',', ' '))
print(f'- Cameras: {len(df_cam):,} lignes'.replace(',', ' '))
print(f'- Commissariats: {len(df_comm):,} lignes'.replace(',', ' '))
print(f'- IRIS: {len(df_iris):,} lignes'.replace(',', ' '))

Fichiers charges:
- Delinquance: 5 238 000 lignes
- Cameras: 1 339 lignes
- Commissariats: 93 lignes
- IRIS: 992 lignes


In [24]:
# 2) Controle rapide
print('Colonnes delinquance:')
print(df_delinq.columns.tolist())

print('\nColonnes cameras:')
print(df_cam.columns.tolist())

print('\nColonnes commissariats:')
print(df_comm.columns.tolist())

print('\nColonnes IRIS:')
print(df_iris.columns.tolist())

print('\nApercu IRIS:')
display(df_iris.head(3))

Colonnes delinquance:
['CODGEO_2025', 'annee', 'indicateur', 'unite_de_compte', 'nombre', 'taux_pour_mille', 'est_diffuse', 'insee_pop', 'insee_pop_millesime', 'insee_log', 'insee_log_millesime', 'complement_info_nombre', 'complement_info_taux']

Colonnes cameras:
['styleUrl', 'icon', 'LOCALISATI', 'NUMÉRO', 'ARRONDISSE', 'MISE_EN_SE', 'lon', 'lat']

Colonnes commissariats:
['name', 'description', 'geometry', 'geo_point_2d']

Colonnes IRIS:
['Geo Shape', 'DEP', 'INSEE_COM', 'NOM_COM', 'IRIS', 'CODE_IRIS', 'NOM_IRIS', 'TYP_IRIS', 'Geo Point', 'ID']

Apercu IRIS:


,Geo Shape,DEP,INSEE_COM,NOM_COM,IRIS,CODE_IRIS,NOM_IRIS,TYP_IRIS,Geo Point,ID
0,"{""coordinates"": [[[2.348444771796439, 48.86237...",75,75101,Paris 1er Arrondissement,205,751010205,Les Halles 5,A,"48.862297570166575, 2.34534858519305",IRIS____0000000751010205
1,"{""coordinates"": [[[2.339143935838259, 48.86220...",75,75101,Paris 1er Arrondissement,303,751010303,Palais Royal 3,A,"48.862970884171986, 2.335627404567633",IRIS____0000000751010303
2,"{""coordinates"": [[[2.295542880656993, 48.87330...",75,75116,Paris 16e Arrondissement,6411,751166411,Chaillot 11,A,"48.871380349816135, 2.296118003770666",IRIS____0000000751166411


In [25]:
# 3) Score risque delinquance (base)
delinq = df_delinq.copy()

# On garde seulement Paris
delinq['codgeo'] = delinq['CODGEO_2025'].astype(str).str.zfill(5)
delinq = delinq[delinq['codgeo'].str.startswith('751')]
delinq['arrondissement'] = delinq['codgeo'].str[-2:]
delinq = delinq[delinq['arrondissement'].isin([f'{i:02d}' for i in range(1, 21)])]

# Taux a utiliser: taux principal, sinon taux complementaire
delinq['taux_effectif'] = pd.to_numeric(delinq['taux_pour_mille'], errors='coerce')
delinq['taux_effectif'] = delinq['taux_effectif'].fillna(pd.to_numeric(delinq['complement_info_taux'], errors='coerce'))

# Colonnes utiles
delinq['annee'] = pd.to_numeric(delinq['annee'], errors='coerce')
delinq['indicateur'] = delinq['indicateur'].astype(str).str.strip()
delinq = delinq.dropna(subset=['annee', 'taux_effectif'])
delinq['annee'] = delinq['annee'].astype(int)

# On passe en format large: 1 colonne par indicateur
pivot = delinq.pivot_table(
    index=['annee', 'arrondissement'],
    columns='indicateur',
    values='taux_effectif',
    aggfunc='mean'
).sort_index()

# Normalisation simple de chaque indicateur
norm = pivot.apply(minmax, axis=0)
score_delinq = norm.mean(axis=1).mul(100).rename('score_risque_delinq_100').reset_index()
score_delinq['score_surete_delinq_100'] = 100 - score_delinq['score_risque_delinq_100']

print(f'Lignes score delinquance: {len(score_delinq):,}'.replace(',', ' '))
display(score_delinq.head(10))

Lignes score delinquance: 200


,annee,arrondissement,score_risque_delinq_100,score_surete_delinq_100
0,2016,01,47.529767,52.470233
1,2016,02,32.582723,67.417277
2,2016,03,16.932099,83.067901
3,2016,04,19.006984,80.993016
4,2016,05,9.227817,90.772183
5,2016,06,12.621852,87.378148
6,2016,07,9.834178,90.165822
7,2016,08,42.658426,57.341574
8,2016,09,20.984063,79.015937
9,2016,10,22.508819,77.491181


In [26]:
# 4) Score par zone geographique (IRIS) puis agregation par arrondissement
iris = df_iris.copy()
iris['codgeo'] = iris['INSEE_COM'].astype(str).str.zfill(5)
iris = iris[iris['codgeo'].str.startswith('751')]
iris['arrondissement'] = iris['codgeo'].str[-2:]
iris = iris[iris['arrondissement'].isin([f'{i:02d}' for i in range(1, 21)])]

# Coordonnees du centre IRIS
geo = iris['Geo Point'].astype(str).str.split(',', n=1, expand=True)
iris['lat'] = pd.to_numeric(geo[0], errors='coerce')
iris['lon'] = pd.to_numeric(geo[1], errors='coerce')
iris = iris.dropna(subset=['lat', 'lon'])

# On rattache le score delinquance de l'arrondissement a chaque IRIS
iris_score = iris[['CODE_IRIS', 'NOM_IRIS', 'arrondissement', 'lat', 'lon']].drop_duplicates().copy()
iris_score = iris_score.merge(score_delinq, on='arrondissement', how='left')

# Cameras: meme score que l'arrondissement, plus simple a lire
iris_score = iris_score.merge(arrondissements[['arrondissement', 'nb_cameras', 'camera_risk_100']], on='arrondissement', how='left')

# Commissariat le plus proche pour chaque IRIS
comm_lat = comm['lat'].to_numpy()
comm_lon = comm['lon'].to_numpy()

def nearest_commissariat(lat, lon):
    if pd.isna(lat) or pd.isna(lon) or len(comm_lat) == 0:
        return np.nan
    dist = distance_km(lat, lon, comm_lat, comm_lon)
    return dist.min()

iris_score['dist_commissariat_km'] = iris_score.apply(lambda r: nearest_commissariat(r['lat'], r['lon']), axis=1)
iris_score['commissariat_risk_100'] = minmax(iris_score['dist_commissariat_km']) * 100

# Score final IRIS
iris_score['score_risque_final_100'] = (0.7 * iris_score['score_risque_delinq_100'] + 0.2 * iris_score['commissariat_risk_100'] + 0.1 * iris_score['camera_risk_100'])
iris_score['score_surete_final_100'] = 100 - iris_score['score_risque_final_100']

# Agregation simple par arrondissement
zone_agregee = iris_score.groupby(['annee', 'arrondissement'], as_index=False).agg(
    score_surete_zone_moyen_100=('score_surete_final_100', 'mean'),
    score_risque_zone_moyen_100=('score_risque_final_100', 'mean'),
    nb_iris=('CODE_IRIS', 'nunique')
)

print(f'Lignes IRIS score: {len(iris_score):,}'.replace(',', ' '))
print(f'Lignes zone aggregee: {len(zone_agregee):,}'.replace(',', ' '))

last_year = int(zone_agregee['annee'].max())
print(f'Derniere annee: {last_year}')
display(zone_agregee[zone_agregee['annee'] == last_year].head(20))

Lignes IRIS score: 9 920
Lignes zone aggregee: 200
Derniere annee: 2025


,annee,arrondissement,score_surete_zone_moyen_100,score_risque_zone_moyen_100,nb_iris
180,2025,01,46.804329,53.195671,17
181,2025,02,65.624306,34.375694,14
182,2025,03,72.928480,27.071520,17
183,2025,04,76.405985,23.594015,21
184,2025,05,80.774861,19.225139,31
185,2025,06,74.782208,25.217792,23
186,2025,07,83.450107,16.549893,36
187,2025,08,66.899838,33.100162,32
188,2025,09,74.136008,25.863992,34
189,2025,10,70.725358,29.274642,42


In [ ]:
# 5) Export
output_dir = Path('data/silver/surete')
output_dir.mkdir(exist_ok=True)

output_path_iris = output_dir / 'kpi_score_surete_iris.csv'
output_path_zone = output_dir / 'kpi_score_surete_arrondissement_agrege_depuis_iris.csv'

iris_score.to_csv(output_path_iris, index=False)
zone_agregee.to_csv(output_path_zone, index=False)

print(f'Fichier cree: {output_path_iris}')
print(f'Fichier cree: {output_path_zone}')
print(f'Nombre de lignes IRIS: {len(iris_score):,}'.replace(',', ' '))
print(f'Nombre de lignes zone aggregee: {len(zone_agregee):,}'.replace(',', ' '))

Fichier cree: output\kpi_score_surete_iris.csv
Fichier cree: output\kpi_score_surete_arrondissement_agrege_depuis_iris.csv
Nombre de lignes IRIS: 9 920
Nombre de lignes zone aggregee: 200
